# Modelling network belief propagation 

## Initialization

Loading the `vis.js` library for data visualization and the network simulation implementation from `network-based.fsx` file.

In [ ]:
#!js
// Load the 'vis' library so that we can access it in generated JS code
req = interactive.configureRequire({
    paths: { vis: "https://visjs.github.io/vis-network/standalone/umd/vis-network.min.js" } });
vis = null; req(["vis"], v => { vis = v; })

In [ ]:
#load "network-based.fsx"
open AgentBased

## Initialization and visualization

In [ ]:
// Generate network with 5 agents and 10 links
let g = Graph.initGraph 5 10

In [ ]:
// Visualize the network using vis.js
Vis.visualizeNetwork g |> HTML

In [ ]:
// For each domain of beliefs, display the network with just beliefs from the domain
[ for domain in ideas -> Vis.visualizeNetwork (Vis.filterGraph domain g) ]
|> String.concat "" |> HTML

## Primitive operations

### Adopting a belief
*"You are friends with Liverpool fans and they convince you to also vote for Greens"*

Find an agent such that it has more than two neighbours that believe in a shared belief (i.e., have links labelled with this belief) from a domain that the given agent does not have any beliefs about and adopt this belief (add edges with the two or more neighbours that beleive it)

In [ ]:
// Single-step operation of the simulation
let update = 
  // Generate all potential (agent, new belief, neightbours) combinations
  Logic.withOne Sim.getAgentsWithBeliefsToAdopt
    // Pick one using 'withOne' and addopt it (add the links)
    Sim.adoptBelief

let t, log = Sim.iterateWithLog 5 update g
String.concat "" (Seq.map Vis.visualizeNetwork t) + log |> HTML

Adding 16<--(C1)-->18 Adding 16<--(C1)-->17 Adding 16<--(C1)-->18 Adding 16<--(C1)-->17 Adding 16<--(C1)-->20

### Creating new connection
*"You randomly run into another fan of The Fugs and you become friends."*

Find a pair of agents that are distinct & disconnected and have a shared belief. Add a new link connecting the two agents.

In [ ]:
let update = 
  Logic.withOne Sim.getDisconnectedCompatibleAgents
    Sim.addEdge

let t, log = Sim.iterateWithLog 5 update g
String.concat "" (Seq.map Vis.visualizeNetwork t) + log |> HTML

Adding 20<--(A3)-->18 Adding 16<--(B1)-->19 Adding 20<--(C1)-->17

### Change a belief

*"You are connected to someone because you both liked LibDems, but all your friends became Tory voters so you change your mind."*

Given any existing edge, look at the beliefs from the same domain of the two agents and change the belief on the edge to any of the possible beliefs. Note that this is done by picking an edge, but picking an agent and then an edge would be the same.

In [ ]:
let update = 
  Logic.withOne Sim.getEdges 
    Sim.updateBeliefOnEdge

let t, log = Sim.iterateWithLog 5 update g
String.concat "" (Seq.map Vis.visualizeNetwork t) + log |> HTML

Adopting A3 on 16<--(A4)-->18 Adopting C1 on 19<--(C1)-->20 Adopting A3 on 16<--(A3)-->18 Adopting C1 on 17<--(C1)-->18 Adopting A3 on 16<--(A3)-->20

### Remove a belief

*"You are connected to someone because of your political opinion, but you keep arguing so you stop caring about politics."*

Find a conflicting edge, i.e., an edge where both agents also believe in other beliefs (are connected via other beliefs) and remove the edge altogether.

In [ ]:
let update = 
  Logic.withOne Sim.getConflictingEdges 
    Sim.removeEdge

let t, log = Sim.iterateWithLog 5 update g
String.concat "" (Seq.map Vis.visualizeNetwork t) + log |> HTML

Removing 16<--(A4)-->18 Removing 17<--(A4)-->19 Removing 19<--(B3)-->20 Removing 16<--(A3)-->20 Removing 16<--(A2)-->17

## Running the simulation

In [ ]:
let update = 
  // Use 'applyOne' to specify that, in each turn, one of the operations
  // should be randomly picked and applied
  Logic.applyOne [
    Logic.withOne Sim.getAgentsWithBeliefsToAdopt
      Sim.adoptBelief
    Logic.withOne Sim.getDisconnectedCompatibleAgents
      Sim.addEdge
    Logic.withOne Sim.getEdges 
      Sim.updateBeliefOnEdge
    Logic.withOne Sim.getConflictingEdges 
      Sim.removeEdge
  ]

// Run 100 iterations of the simulation
let t, log = Sim.iterateWithLog 100 update g
// Draw every 20th iteration (by choosing networks at index i*20 from the trace 't')
String.concat "" [ for i in 0 .. 5 -> Vis.visualizeNetwork t.[i*20] ] + log |> HTML

Removing 3<--(A2)-->5 Removing 2<--(C4)-->4 Adopting C4 on 1<--(C1)-->4 Adding 2<--(C4)-->3 Adding 2<--(C4)-->3 Adding 1<--(A1)-->3 Adding 1<--(A1)-->3 Adding 4<--(A1)-->1 Adding 4<--(A1)-->1 Adding 5<--(A1)-->4 Adding 5<--(A1)-->3 Removing 3<--(C3)-->5 Adding 2<--(A1)-->1 Adding 2<--(C4)-->4 Adding 5<--(B2)-->4 Adding 5<--(B2)-->4 Adding 3<--(A1)-->4 Adopting A1 on 1<--(A1)-->2 Removing 2<--(C4)-->3 Removing 1<--(C4)-->3 Adopting B1 on 2<--(B1)-->3 Adopting A1 on 4<--(A1)-->5 Adding 1<--(A1)-->5 Removing 2<--(C4)-->4 Adding 2<--(C1)-->1 Adding 2<--(C1)-->3 Adding 2<--(C1)-->3 Adding 5<--(A1)-->2 Adopting A1 on 2<--(A1)-->3 Removing 1<--(C1)-->2 Adding 4<--(A1)-->2 Removing 4<--(C3)-->5 Removing 1<--(C4)-->4 Adopting A1 on 3<--(A1)-->4 Adopting A1 on 1<--(A1)-->2 Adopting A1 on 3<--(A1)-->5 Adopting A1 on 1<--(A1)-->3 Adopting B2 on 1<--(B2)-->4 Adding 4<--(C1)-->2 Adding 4<--(C1)-->3 Adding 4<--(C1)-->1 Adding 4<--(C1)-->1 Adding 4<--(C1)-->1 Adding 5<--(C1)-->2 Adding 5<--(C1)-->1 Adding 5<--(C1)-->4 Adding 5<--(C1)-->4 Adding 5<--(C1)-->4 Adding 5<--(C1)-->3 Adopting C1 on 2<--(C1)-->4 Adopting C1 on 4<--(C1)-->5 Adopting A1 on 1<--(A1)-->4 Adopting C1 on 1<--(C1)-->4 Adopting A1 on 2<--(A1)-->3 Adopting B2 on 4<--(B2)-->5 Adopting C1 on 1<--(C1)-->3 Adopting A1 on 1<--(A1)-->5 Adopting C1 on 2<--(C1)-->5 Adopting C1 on 3<--(C1)-->5 Adopting C1 on 2<--(C1)-->4 Adopting C1 on 4<--(C1)-->5 Adopting C1 on 1<--(C1)-->5 Adopting B2 on 4<--(B2)-->5 Adopting A1 on 2<--(A1)-->4

In [ ]:
// The above seems to be adding way too many edges - so to ballance that
// we can use 'applyOneProb' that lets us specify probability (set this 
// to higher for removing and edge) and we can also add a new rule using
// Sim.getEdges and Sim.removeEdge to just randomly remove an edge.
// If agent has no connection, none of the standard operations ever add
// it back to the network, so the below adds one more operation, which
// randomly connects disconnected agent to random agent in the network
let update = 
  Logic.applyOneProb [
    0.1, Logic.withOne Sim.getAgentsWithBeliefsToAdopt
      Sim.adoptBelief
    0.1, Logic.withOne Sim.getDisconnectedCompatibleAgents
      Sim.addEdge
    0.3, Logic.withOne Sim.getEdges 
      Sim.updateBeliefOnEdge
    0.3, Logic.withOne Sim.getConflictingEdges 
      Sim.removeEdge
    // Randomly remove edge to keep the average number of edges
    0.1, Logic.withOne Sim.getEdges 
      Sim.removeEdge
    // Connect agent that has been completely disconnected
    0.1, Logic.withOne Sim.getDisconnectedAgents (fun a1 ->
      Logic.withOne Sim.getConnectedAgents (fun a2 ->
        Logic.withOne (Sim.getAgentBeliefs a2.ID) (fun belief ->
          Sim.addEdge (a1.ID, a2.ID, belief))))
  ]

// Run 100 iterations of the simulation
let t, log = Sim.iterateWithLog 100 update g
// Draw every 20th iteration (by choosing networks at index i*20 from the trace 't')
String.concat "" [ for i in 0 .. 5 -> Vis.visualizeNetwork t.[i*20] ] + log |> HTML

Adopting A3 on 4<--(A1)-->5 Adding 5<--(C4)-->1 Adding 5<--(C4)-->1 Adding 5<--(C4)-->2 Adopting A3 on 4<--(A3)-->5 Adopting C2 on 3<--(C2)-->4 Removing 1<--(B4)-->5 Adding 5<--(B3)-->2 Adding 5<--(B3)-->4 Adding 5<--(B3)-->2 Removing 1<--(C4)-->2 Adopting A3 on 4<--(A3)-->5 Adopting C2 on 2<--(C3)-->4 Removing 2<--(C4)-->5 Removing 1<--(C4)-->5 Adopting B2 on 1<--(B2)-->3 Removing 1<--(A3)-->2 Adopting A3 on 4<--(A3)-->5 Removing 1<--(B2)-->3 Adding 2<--(A3)-->1 Adopting A3 on 1<--(A3)-->5 Adding 5<--(C2)-->2 Adding 5<--(C2)-->4 Adding 5<--(C2)-->2 Adding 5<--(C2)-->4 Adding 5<--(C2)-->2 Adding 1<--(B3)-->2 Adding 1<--(B3)-->5 Adopting B3 on 2<--(B3)-->5 Removing 2<--(B3)-->4 Adopting C2 on 2<--(C2)-->5 Adopting A3 on 1<--(A3)-->5 Adopting C2 on 3<--(C2)-->4 Removing 2<--(A3)-->5 Adopting B3 on 1<--(B3)-->5 Adding 1<--(C2)-->2 Adding 1<--(C2)-->5 Adding 1<--(C2)-->2 Adding 1<--(C2)-->5 Adopting C2 on 2<--(C2)-->5 Adding 4<--(B3)-->1 Adopting B3 on 1<--(B3)-->5 Adopting C2 on 2<--(C2)-->5 Adopting C2 on 2<--(C2)-->5 Adopting C2 on 2<--(C2)-->4 Adopting B3 on 4<--(B3)-->5 Adding 3<--(C2)-->5 Adopting A3 on 1<--(A3)-->5 Adding 1<--(C2)-->3 Adopting C2 on 1<--(C2)-->5 Adopting B3 on 2<--(B3)-->5 Removing 3<--(C2)-->5 Adding 3<--(A3)-->1 Adding 3<--(A3)-->4 Adopting C2 on 1<--(C2)-->5 Adopting B3 on 2<--(B3)-->5 Adding 3<--(B3)-->1 Adding 3<--(B3)-->4 Adding 3<--(B3)-->1 Adding 3<--(B3)-->4 Adopting B3 on 3<--(B3)-->4 Adopting C2 on 2<--(C2)-->4 Adopting A3 on 3<--(A3)-->4 Adopting C2 on 3<--(C2)-->4 Removing 3<--(A3)-->4 Adopting B3 on 4<--(B3)-->5 Removing 2<--(C2)-->5 Adding 2<--(B3)-->3 Adding 5<--(B3)-->3 Removing 1<--(B3)-->3 Removing 1<--(C2)-->2 Adopting B3 on 1<--(B3)-->4 Adopting B3 on 2<--(B3)-->5 Removing 4<--(C2)-->5 Removing 1<--(C2)-->5 Adopting B3 on 2<--(B3)-->5 Adding 5<--(C2)-->3 Adding 5<--(C2)-->1 Adding 5<--(C2)-->2 Adding 5<--(C2)-->4 Adding 5<--(C2)-->2 Adding 5<--(C2)-->1 Adding 5<--(C2)-->4 Adopting C2 on 1<--(C2)-->3 Adopting B3 on 3<--(B3)-->5

## Calculating network statistics

In [ ]:
#r "nuget: Plotly.NET"
#r "nuget: Plotly.NET.Interactive"

Installed Packages Plotly.NET, 2.0.0 Plotly.NET.Interactive, 2.0.0

In [ ]:
// The 'iterate' function returns a sequence of network states 
// so we can iterate over that to calculate whetever stats
// we are interested in. Note that this is lazy sequence.
// Using the 'update' function from the previous
let t = Sim.iterate 100 update g

let averageDegree (g:Graph) = 
  g.Agents 
  |> Seq.map (fun a -> 
    let neighbours = Graph.getEdges a.ID g 
    float (Seq.length neighbours) ) 
  |> Seq.average

t |> Seq.map averageDegree |> Seq.indexed |> Chart.Line

<!-- Plotly chart will be drawn inside this DIV --> 


 
</div